# Machine Learning Forecasting

Use calendar and lag features in chronological folds. Imputers, encoders, scaling, and selection belong inside each training fold.


In [1]:
# PROJECT_ROOT_BOOTSTRAP: support kernels started from the repository root or notebooks/.
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Could not locate the project root containing src/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from src.data import load_demand, TARGETS
from src.features import make_calendar_features, make_lag_features

demand = load_demand("data/raw/clientes_restaurantes.csv")
for target in TARGETS:
    y = demand[target]
    X = make_calendar_features(y.index).join(make_lag_features(y))
    valid = y.notna() & X.notna().all(axis=1)
    model = make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingRegressor(random_state=42))
    print(target, "eligible one-step rows:", int(valid.sum()), "of", len(y))
    # Fit and score only inside chronological folds; recursive forecasting is required for multi-step deployment.

Chimi & Churri eligible one-step rows: 418 of 478
Assa Frão eligible one-step rows: 434 of 478
